In [3]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA

In [4]:
!cp /kaggle/input/datasets/anamikapatel8/croma-base/CROMA_base.pt /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/pretrain_croma.py /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/use_croma.py /kaggle/working/CROMA/


In [5]:
!pip install rasterio

In [ ]:
import os
import json

S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"

def get_chip_id(folder_name):
    # Extracting the coordinate suffix
    return folder_name.split("_")[-1]

def build_file_list():
    if not os.path.exists(S1_ROOT) or not os.path.exists(S2_ROOT):
        print(f"ERROR: Root paths not found. Please verify S1_ROOT and S2_ROOT.")
        return []

    s1_folders = sorted(os.listdir(S1_ROOT))
    s2_folders = sorted(os.listdir(S2_ROOT))
    s2_dict = {get_chip_id(f): f for f in s2_folders}
    
    pairs = []
    
    # Required files for a research pair
    s1_required = ["VV.tif", "VH.tif", "LabelWater.tif"]
    # CROMA expects 12 bands 
    s2_required = [f"B{i}.tif" for i in range(1, 10)] + ["B8A.tif", "B11.tif", "B12.tif"]

    print("--- Starting File Integrity Audit ---")

    for s1_f in s1_folders:
        cid = get_chip_id(s1_f)
        if cid in s2_dict:
            s1_path = os.path.join(S1_ROOT, s1_f)
            s2_path = os.path.join(S2_ROOT, s2_dict[cid])
            
            # Verifying SAR files exist
            s1_valid = all(os.path.exists(os.path.join(s1_path, f)) for f in s1_required)
            
            # Verifying Optical files exist
            s2_valid = all(os.path.exists(os.path.join(s2_path, f)) for f in s2_required)
            
            if s1_valid and s2_valid:
                pairs.append((s1_f, s2_dict[cid]))

    print(f"Total S1 folders: {len(s1_folders)}")
    print(f"Total S2 folders: {len(s2_folders)}")
    print(f"Successfully matched and verified pairs: {len(pairs)}")
    
    if len(pairs) > 0:
        print(f"\nExample Matched Pair:")
        print(f" S1: {pairs[100][0]}")
        print(f" S2: {pairs[100][1]}")
    
    return pairs

matched_pairs = build_file_list()

# Saving to a JSON file so train.py can read it
with open("/kaggle/working/matched_pairs.json", "w") as f:
    json.dump(matched_pairs, f)

print(f"Successfully saved {len(matched_pairs)} pairs to matched_pairs.json")

--- Starting File Integrity Audit ---
Total S1 folders: 900
Total S2 folders: 900
Successfully matched and verified pairs: 900

Example Matched Pair:
 S1: S1A_IW_GRDH_1SDV_20180507T160424_20180507T160449_021800_025A09_981B_01536-10752
 S2: 20180507T074611_20180507T080728_T36NXF_01536-10752
Successfully saved 900 pairs to matched_pairs.json


In [ ]:
%%writefile /kaggle/working/code/dataset_c2sms.py
import sys
sys.path.append("/kaggle/working/code")
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import rasterio

# Refined Lee filter 
def refined_lee_filter(img, size=7):
    from scipy.ndimage import uniform_filter
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img_mean = uniform_filter(img, (size, size))
    img_sqr_mean = uniform_filter(img**2, (size, size))
    img_variance = np.maximum(0, img_sqr_mean - img_mean**2)
    overall_mean = np.mean(img)
    if overall_mean == 0: return img
    overall_variance = np.var(img)
    denominator = img_variance + (overall_variance / (overall_mean**2 + 1e-6))
    img_weights = np.clip(img_variance / (denominator + 1e-6), 0, 1)
    return img_mean + img_weights * (img - img_mean)


IMG_SIZE = 224   # CROMA and SARATR-X positional embeddings fixed at 224×224


class C2SMSDataset(Dataset):
    def __init__(self, s1_root, s2_root, matched_pairs):
        self.s1_root  = s1_root
        self.s2_root  = s2_root
        self.pairs    = matched_pairs
        self.eo_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

    def __len__(self):
        return len(self.pairs)

    def _preprocess_sar(self, sar):
        # CHANGE 1: Lee filter call removed — raw dB normalisation only (~1μs vs ~30ms)
        sar_db = 10.0 * np.log10(np.clip(sar, 1e-8, None))
        p1, p99 = np.percentile(sar_db, 1), np.percentile(sar_db, 99)
        sar_db = np.clip(sar_db, p1, p99)
        return ((sar_db - p1) / (p99 - p1 + 1e-6) * 2.0 - 1.0).astype(np.float32)

    def _preprocess_eo(self, eo):
        eo = np.nan_to_num(eo, nan=0.0)
        return (np.clip(eo, 0, 10000) / 10000.0 * 2.0 - 1.0).astype(np.float32)

    def __getitem__(self, idx):
        s1_folder, s2_folder = self.pairs[idx]
        s1_p = os.path.join(self.s1_root, s1_folder)
        s2_p = os.path.join(self.s2_root, s2_folder)

        # --- SAR ---
        with rasterio.open(os.path.join(s1_p, "VV.tif")) as src:
            vv = self._preprocess_sar(src.read(1))
        with rasterio.open(os.path.join(s1_p, "VH.tif")) as src:
            vh = self._preprocess_sar(src.read(1))
        sar_tensor = torch.from_numpy(np.stack([vv, vh], axis=0))  # (2, H, W)

        # --- Labels & cloud mask ---
        with rasterio.open(os.path.join(s1_p, "LabelWater.tif")) as src:
            s1_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelWater.tif")) as src:
            s2_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelCloud.tif")) as src:
            cloud = torch.from_numpy((src.read(1) == 1).astype(np.float32))

        # --- MS ---
        eo_data = [
            self._preprocess_eo(rasterio.open(os.path.join(s2_p, f"{b}.tif")).read(1))
            for b in self.eo_bands
        ]
        eo_tensor = torch.from_numpy(np.stack(eo_data, axis=0))    # (12, H, W)

        # --- Resizing all tensors to 224×224 ---
        sar_tensor = F.interpolate(
            sar_tensor.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
            mode='bilinear', align_corners=False
        ).squeeze(0)                                                # (2, 224, 224)

        eo_tensor = F.interpolate(
            eo_tensor.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
            mode='bilinear', align_corners=False
        ).squeeze(0)                                                # (12, 224, 224)

        s1_label = F.interpolate(
            s1_label.unsqueeze(0).unsqueeze(0).float(),
            size=(IMG_SIZE, IMG_SIZE), mode='nearest'
        ).squeeze(0).squeeze(0).long()                              # (224, 224)

        s2_label = F.interpolate(
            s2_label.unsqueeze(0).unsqueeze(0).float(),
            size=(IMG_SIZE, IMG_SIZE), mode='nearest'
        ).squeeze(0).squeeze(0).long()                              # (224, 224)

        cloud = F.interpolate(
            cloud.unsqueeze(0).unsqueeze(0),
            size=(IMG_SIZE, IMG_SIZE), mode='nearest'
        ).squeeze(0).squeeze(0)                                     # (224, 224)

        return sar_tensor, eo_tensor, s1_label, s2_label, cloud

Writing /kaggle/working/code/dataset_c2sms.py


In [ ]:
%%writefile /kaggle/working/code/model.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedFusion(nn.Module):
    """
    Context-Aware Gated Fusion (CAGF) — global scalar gate via MLP on GAP vectors.

    CAGF formulation:
        z   = [GAP(SAR_feats) ; GAP(EO_feats) ; cloud_ratio]   ∈ R^(2*dim+1)
        g   = σ( W2 · ReLU( W1·z + b1 ) + b2 )                scalar
        out = g · SAR_feats + (1-g) · EO_feats                 convex blend

    Args:
        dim : feature dimension of both SAR and EO token maps 
    """
    def __init__(self, dim):
        super().__init__()
        # MLP gate: input is [GAP(SAR) | GAP(EO) | cloud_ratio] = 2*dim+1
        self.gate = nn.Sequential(
            nn.Linear(dim * 2 + 1, dim),
            nn.ReLU(),
            nn.Linear(dim, 1),
            nn.Sigmoid()
        )

    def forward(self, sar_feats, eo_feats, cloud_ratio):
        """
        Args:
            sar_feats   : (B, dim, h, w)  — SAR spatial feature map
            eo_feats    : (B, dim, h, w)  — EO  spatial feature map
            cloud_ratio : (B,)            — per-image cloud cover fraction [0,1]
        Returns:
            fused       : (B, dim, h, w)  — blend of SAR and EO features
        """
        B = sar_feats.shape[0]

        # Global Average Pool each feature map to a vector
        sar_vec = F.adaptive_avg_pool2d(sar_feats, 1).view(B, -1)        # (B, dim)
        eo_vec  = F.adaptive_avg_pool2d(eo_feats,  1).view(B, -1)        # (B, dim)

        # Concatenate with scalar cloud cover ratio
        z = torch.cat([sar_vec, eo_vec, cloud_ratio.view(B, 1)], dim=1)  # (B, 2*dim+1)

        # Scalar gate: one value per image, broadcast over spatial dims
        g = self.gate(z).view(B, 1, 1, 1)                                # (B, 1, 1, 1)

        # Convex weighted blend 
        return g * sar_feats + (1 - g) * eo_feats                        # (B, dim, h, w)


# --- FCNN baseline---
class SimpleFCNN(nn.Module):
    def __init__(self, sar_channels=0, eo_channels=0, out_channels=2, initial_features=64, modality='eo_only'):
        super().__init__()
        self.modality = modality

        if modality == 'sar_only':
            self.total_in_channels = sar_channels + 1
        elif modality == 'eo_only':
            self.total_in_channels = eo_channels + 1
        elif modality == 'joint':
            self.total_in_channels = sar_channels + eo_channels + 1
        else:
            raise ValueError(f"Unknown FCNN modality: {modality}")

        self.encoder = nn.Sequential(
            nn.Conv2d(self.total_in_channels, initial_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(initial_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(initial_features, initial_features * 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(initial_features * 2),
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(initial_features * 2, initial_features, kernel_size=2, stride=2),
            nn.BatchNorm2d(initial_features),
            nn.ReLU(inplace=True),
            nn.Conv2d(initial_features, out_channels, kernel_size=1)
        )

    def forward(self, sar=None, eo=None, cloud=None):
        inputs_to_concat = []
        cloud_expanded = cloud.unsqueeze(1).float()

        if self.modality == 'sar_only' or self.modality == 'joint':
            if sar is None: raise ValueError("SAR input required.")
            inputs_to_concat.append(sar)

        if self.modality == 'eo_only' or self.modality == 'joint':
            if eo is None: raise ValueError("EO input required.")
            inputs_to_concat.append(eo)

        inputs_to_concat.append(cloud_expanded)
        return self.decoder(self.encoder(torch.cat(inputs_to_concat, dim=1)))


class CROMASegmentation(nn.Module):
    def __init__(self, croma_model, feat_dim, patch_size, mode="joint",
                 model_type="croma", fcnn_modality='joint'):
        super().__init__()
        self.model_type = model_type

        if self.model_type == "croma":
            self.croma      = croma_model
            self.feat_dim   = feat_dim
            self.patch_size = patch_size
            self.mode       = mode

            if mode == "joint":
                self.fusion = GatedFusion(feat_dim)

            # Segmentation head 
            self.head = nn.Sequential(
                nn.Conv2d(feat_dim, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.Conv2d(128, 2, kernel_size=1)
            )

            for p in self.croma.parameters():
                p.requires_grad = False

        elif self.model_type == "fcnn":
            self.fcnn = SimpleFCNN(sar_channels=2, eo_channels=12,
                                   out_channels=2, modality=fcnn_modality)
        else:
            raise ValueError("model_type must be 'croma' or 'fcnn'")

    def forward(self, sar=None, eo=None, cloud=None):
        if self.model_type == "croma":
            B, _, H, W = eo.shape if eo is not None else sar.shape
            h, w = H // self.patch_size, W // self.patch_size

            # CROMA backbone (frozen)
            with torch.no_grad():
                out = self.croma(SAR_images=sar, optical_images=eo)

            if self.mode == "joint":
                s_feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)
                e_feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

                # Scalar cloud cover ratio per image: mean over spatial pixels
                # cloud shape: (B, H, W) -> (B,) scalar
                cloud_ratio = cloud.float().mean(dim=(1, 2))  # (B,)

                # CAGF: global scalar gate, convex blend
                feats = self.fusion(s_feats, e_feats, cloud_ratio)

            elif self.mode == "eo":
                feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

            else:  # "sar"
                feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

            feats = F.interpolate(feats, size=(H, W), mode="bilinear", align_corners=False)
            return self.head(feats)

        elif self.model_type == "fcnn":
            return self.fcnn(sar=sar, eo=eo, cloud=cloud)

Writing /kaggle/working/code/model.py


In [9]:
%%writefile /kaggle/working/code/croma_wrapper.py 
import sys
sys.path.append("/kaggle/working")
import torch
from CROMA.use_croma import PretrainedCROMA

def load_croma_encoder(pretrained_path, image_resolution, mode="joint", device="cpu"):
    """
    Loads CROMA backbone based on requested modality.
    mode: 'joint' (both), 'eo' (optical), or 'sar' (SAR)
    """
    modality_map = {"joint": "both", "eo": "optical", "sar": "SAR"}
    
    model = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size="base",
        modality=modality_map[mode],
        image_resolution=image_resolution
    ).to(device)
    
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
        
    return model, model.encoder_dim, model.patch_size

Writing /kaggle/working/code/croma_wrapper.py


In [ ]:
%%writefile /kaggle/working/code/train_UP.py
import sys
sys.path.append("/kaggle/working/code")
import os, json, random
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from dataset_c2sms import C2SMSDataset
from model import CROMASegmentation
from croma_wrapper import load_croma_encoder


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


# --- Configuration ---
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CROMA_CKPT = "/kaggle/working/CROMA/CROMA_base.pt"
S1_ROOT    = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT    = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
PAIRS_JSON = "/kaggle/working/matched_pairs.json"

RESUME_PATH     = "/kaggle/input/datasets/anamikapatel8/croma-sar-eo-fused-checkpoint/croma_fused_checkpoint_UP.pth"
SAVE_PATH_MODEL = "/kaggle/working/croma_c2sms_fused_UP.pt"
SAVE_PATH_CKPT  = "/kaggle/working/croma_fused_checkpoint_UP.pth"


# --- Metrics ---
def calculate_iou(preds, target):
    smooth = 1e-6
    inter  = (preds & target).sum().float()
    union  = (preds | target).sum().float()
    return (inter + smooth) / (union + smooth)

def calculate_dice_score(preds, target):
    smooth = 1e-6
    inter  = (preds * target).sum().float()
    return (2. * inter + smooth) / (preds.sum().float() + target.sum().float() + smooth)


# --- Cloud-Aware Loss ---
def cloud_aware_loss(logits, s1_label, s2_label, cloud_mask, criterion):
    target   = (s2_label * (1 - cloud_mask.long())) + (s1_label * cloud_mask.long())
    ce       = criterion(logits, target)
    probs    = torch.softmax(logits, dim=1)[:, 1]
    target_f = target.float()
    inter    = (probs * target_f).sum(dim=(1, 2))
    union    = probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
    dice     = (1 - (2 * inter + 1e-6) / (union + 1e-6)).mean()
    return 0.3 * ce + 0.7 * dice


# --- Single training run ---
def train_model(seed=42):
    set_seed(seed)

    with open(PAIRS_JSON, "r") as f:
        matched_pairs = json.load(f)

    dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
    indices = torch.randperm(len(dataset),
                             generator=torch.Generator().manual_seed(seed)).tolist()
    split   = int(0.8 * len(dataset))
    train_idx, val_idx = indices[:split], indices[split:]

    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=8,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Subset(dataset, val_idx),   batch_size=8,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f">>> Initializing CAGF model (seed={seed})...")
    
    croma, f_dim, p_size = load_croma_encoder(CROMA_CKPT, 224, mode="joint", device=DEVICE)
    model = CROMASegmentation(croma, f_dim, p_size, mode="joint").to(DEVICE)

    optimizer = torch.optim.Adam([
        {'params': model.fusion.parameters(), 'lr': 1e-4},
        {'params': model.head.parameters(),   'lr': 5e-5}
    ])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 12.5]).to(DEVICE))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    # --- Checkpoint resume ---
    start_epoch, best_iou, best_dice = 0, 0.0, 0.0

    if os.path.exists(RESUME_PATH):
        print(f">>> Resuming from checkpoint: {RESUME_PATH}")
        ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_iou    = ckpt.get('best_iou',  0.0)
        best_dice   = ckpt.get('best_dice', 0.0)
        print(f">>> Resuming from Epoch {start_epoch} | "
              f"Best IoU: {best_iou:.4f} | Best Dice: {best_dice:.4f}")
    else:
        print(">>> No checkpoint found. Starting from scratch.")

    # --- Training loop ---
    for epoch in range(start_epoch, 50):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch:02d} [CROMA | CAGF | seed={seed}]")

        for sar, eo, s1_l, s2_l, cloud in pbar:
            sar, eo, s1_l, s2_l, cloud = [
                x.to(DEVICE, non_blocking=True)
                for x in [sar, eo, s1_l, s2_l, cloud]
            ]
            optimizer.zero_grad()
            logits = model(sar=sar, eo=eo, cloud=cloud)
            loss   = cloud_aware_loss(logits, s1_l, s2_l, cloud, criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.fusion.parameters()) + list(model.head.parameters()),
                max_norm=1.0
            )
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}",
                              "lr":   f"{optimizer.param_groups[0]['lr']:.2e}"})

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # --- Validation with ±std ---
        model.eval()
        val_ious, val_dices = [], []
        with torch.no_grad():
            for sar, eo, s1_l, s2_l, cloud in val_loader:
                sar, eo, s1_l, s2_l, cloud = [
                    x.to(DEVICE, non_blocking=True)
                    for x in [sar, eo, s1_l, s2_l, cloud]
                ]
                logits = model(sar=sar, eo=eo, cloud=cloud)
                preds  = (torch.softmax(logits, dim=1)[:, 1] > 0.4).long()
                target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())

                val_ious.append(calculate_iou(preds, target))
                val_dices.append(calculate_dice_score(preds, target))

        avg_iou  = torch.mean(torch.stack(val_ious)).item()
        std_iou  = torch.std(torch.stack(val_ious)).item()
        avg_dice = torch.mean(torch.stack(val_dices)).item()
        std_dice = torch.std(torch.stack(val_dices)).item()

        if avg_dice > best_dice:
            best_dice = avg_dice

        print(f"Epoch {epoch:02d} | LR: {current_lr:.2e} | "
              f"Loss: {epoch_loss/len(train_loader):.4f} | "
              f"Val IoU: {avg_iou:.4f} +/- {std_iou:.4f} | "
              f"Val Dice: {avg_dice:.4f} +/- {std_dice:.4f}")

        if avg_iou > best_iou:
            best_iou = avg_iou
            torch.save(model.state_dict(), SAVE_PATH_MODEL)
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_iou':             best_iou,
                'best_dice':            best_dice,
            }, SAVE_PATH_CKPT)
            print(f"    >>> Saved Best Model + Checkpoint  "
                  f"(IoU: {best_iou:.4f}, Dice: {best_dice:.4f})")

    return best_iou, best_dice


# --- Multi-seed runner ---
def run_multiple_trainings(num_runs=3, start_seed=0):
    print(f"\n>>> Running {num_runs} training experiments with different seeds...")
    all_ious, all_dices = [], []

    for i in range(num_runs):
        seed = start_seed + i
        print(f"\n--- Run {i+1}/{num_runs} | Seed: {seed} ---")
        best_iou, best_dice = train_model(seed=seed)
        all_ious.append(best_iou)
        all_dices.append(best_dice)
        print(f"--- Run {i+1} complete | "
              f"Best IoU: {best_iou:.4f} | Best Dice: {best_dice:.4f} ---")

    mean_iou  = np.mean(all_ious)
    std_iou   = np.std(all_ious)
    mean_dice = np.mean(all_dices)
    std_dice  = np.std(all_dices)

    print(f"\n>>> All {num_runs} runs completed.")
    print(f"Mean Best IoU  : {mean_iou:.4f} ± {std_iou:.4f}")
    print(f"Mean Best Dice : {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"Individual IoUs : {[f'{v:.4f}' for v in all_ious]}")
    print(f"Individual Dices: {[f'{v:.4f}' for v in all_dices]}")


if __name__ == '__main__':
    run_multiple_trainings(num_runs=3, start_seed=0)

Writing /kaggle/working/code/train_UP.py


In [ ]:
!python /kaggle/working/code/train_UP.py

In [ ]:
%%writefile /kaggle/working/code/find_optimal_weights.py
import sys
sys.path.append("/kaggle/working/code")
import os, json, torch, random
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset_c2sms import C2SMSDataset
from model import CROMASegmentation
from croma_wrapper import load_croma_encoder

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROMA_CKPT = "/kaggle/working/CROMA/CROMA_base.pt"
S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
PAIRS_JSON = "/kaggle/working/matched_pairs.json"

def get_dataset_class_ratio(dataset):
    print(f"Calculating class ratio from ALL {len(dataset)} samples...")
    water_pixels = 0
    total_pixels = 0
    
    # looping through every single image in the dataset
    for i in range(len(dataset)):
        _, _, s1_l, _, _ = dataset[i]
        water_pixels += torch.sum(s1_l == 1).item()
        total_pixels += s1_l.numel()
    
    land_pixels = total_pixels - water_pixels
    ratio = land_pixels / (water_pixels + 1e-6)
    return ratio

# --- 2. Metric for Grid search ---
def evaluate_weight_combo(model, loader, class_weight):
    model.eval()
    ious = []
    with torch.no_grad():
        for sar, eo, s1_l, s2_l, cloud in loader:
            sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE) for x in [sar, eo, s1_l, s2_l, cloud]]
            target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())
            
            logits = model(sar=sar, eo=eo, cloud=cloud)
            
            # --- Applying the class_weight to the water channel  ---
            # This shifts the model's bias towards water based on the search parameter
            logits[:, 1, :, :] *= class_weight 
            
            preds = (torch.softmax(logits, dim=1)[:, 1] > 0.5).long()
            
            intersection = (preds & target).sum().float()
            union = (preds | target).sum().float()
            ious.append((intersection + 1e-6) / (union + 1e-6))
            
    return torch.mean(torch.stack(ious)).item()

# --- Main Search ---
with open(PAIRS_JSON, "r") as f:
    matched_pairs = json.load(f)

dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
ratio = get_dataset_class_ratio(dataset)
print(f"Detected Land-to-Water Pixel Ratio: {ratio:.2f}")

# Search Space
class_weights = [0.3, 0.5, 0.7, 1.0] 
ce_dice_ratios = [(0.5, 0.5)]

# Load Model once
croma, f_dim, p_size = load_croma_encoder(CROMA_CKPT, 512, mode="joint", device=DEVICE)
model = CROMASegmentation(croma, f_dim, p_size, mode="joint").to(DEVICE)
model.load_state_dict(torch.load("/kaggle/input/datasets/anamikapatel8/croma-c2sms-eo-sar/croma_c2sms_gated_best_1.pt", map_location=DEVICE))

# Full validation subset (Last 20% of the dataset)
indices = list(range(len(dataset)))
split = int(0.8 * len(dataset))
val_loader = DataLoader(Subset(dataset, indices[split:]), batch_size=4)

print(f"\n--- Starting Grid Search on {len(indices)-split} validation images ---")
best_score = 0
best_params = {}

for cw in class_weights:
    for ce_w, dice_w in ce_dice_ratios:
        # evaluate the current model to see how it performs with these weights
        score = evaluate_weight_combo(model, val_loader, cw)
        print(f"Testing ClassWeight: {cw} | CE/Dice Ratio: {ce_w}/{dice_w} | Mean IoU: {score:.4f}")
        
        if score > best_score:
            best_score = score
            best_params = {"class_weight": cw, "ce_weight": ce_w, "dice_weight": dice_w}

print("\n" + "="*30)
print("OPTIMAL PARAMETERS FOUND:")
print(f"Class Weight: {best_params['class_weight']}")
print(f"CE Weight:    {best_params['ce_weight']}")
print(f"Dice Weight:  {best_params['dice_weight']}")
print("="*30)